# PYNQ Latency Benchmark

Runs `run_pynq_rtt.py`, saves JSON/CSV, and displays key stats.

## What is measured

**`rtt`** — dedicated UDP echo probe: board sends a ping packet to the EC2 server, server replies immediately, board measures round-trip time. Measures raw network path only; no game logic, no BRAM write.

**`button_to_visible`** — full board-to-server-to-board path: starts timing on a new button press edge, sends one `PKT_STATE_UPDATE` to the server (exactly as normal gameplay does), then stops timing when the next `PKT_GAME_STATE` reply arrives and the pose is written to BRAM. This is the real end-to-end latency a player experiences: button press → server processes → board receives updated state → raycaster sees new position.

## Trigger modes (rtt only)
- `auto`: probes sent back-to-back
- `button`: one probe per new button press edge

## What is not measured
- Monitor/browser rendering delay
- `button_to_visible` sends one packet per sample (press edge, not hold) — the player moves one step on the monitor canvas per press, not continuously

## Key stats for the report
- `p95_rtt_ms` — network stability headline
- `button_to_visible_p95_ms` — worst-case button-to-BRAM latency (includes server round trip)

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Resolve the folder that contains this notebook / script, regardless of
# what Jupyter's working directory happens to be.
try:
    # VSCode / nbconvert set __vsc_ipynb_file__; standard Jupyter sets __file__
    _nb_file = globals().get("__vsc_ipynb_file__") or globals().get("__file__") or ""
    NOTEBOOK_DIR = Path(_nb_file).resolve().parent if _nb_file else Path.cwd()
except Exception:
    NOTEBOOK_DIR = Path.cwd()

# run_pynq_rtt.py lives in the same folder as this notebook
SCRIPT_DIR = NOTEBOOK_DIR
assert (SCRIPT_DIR / "run_pynq_rtt.py").exists(), (
    f"run_pynq_rtt.py not found in {SCRIPT_DIR} — "
    "make sure the notebook is inside the pynq_client_tests folder"
)
print(f"SCRIPT_DIR: {SCRIPT_DIR}")

## Configure Run

In [ ]:
SERVER = "3.9.71.204"
PORT = 9000
LABEL = "idle"
MEASURE = "rtt"  # use "button_to_visible" for button press to returned server-state timing
TRIGGER = "auto"  # in rtt mode, use "button" for one RTT sample per new board button press
SAMPLES = 100
TIMEOUT_S = 1.0

DATA_DIR = SCRIPT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / f"udp_rtt_{LABEL}.csv"
JSON_PATH = DATA_DIR / f"udp_rtt_{LABEL}.json"

SERVER, PORT, LABEL, MEASURE, TRIGGER, SAMPLES, TIMEOUT_S, CSV_PATH, JSON_PATH

## Run Benchmark

This runs the same command you would use in the terminal.

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_DIR / "run_pynq_rtt.py"),
    "--server", SERVER,
    "--port", str(PORT),
    "--label", LABEL,
    "--measure", MEASURE,
    "--trigger", TRIGGER,
    "--samples", str(SAMPLES),
    "--timeout", str(TIMEOUT_S),
    "--csv-out", str(CSV_PATH),
    "--json-out", str(JSON_PATH),
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(SCRIPT_DIR), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"Benchmark exited with code {result.returncode}")

## Load Stats

In [ ]:
report = json.loads(JSON_PATH.read_text())
report

## Key Metrics

In [ ]:
if report.get("measurement") == "button_to_visible":
    stats = report.get("button_to_visible_stats") or {}
    summary = {
        "label": report.get("label"),
        "measurement": report.get("measurement"),
        "trigger": report.get("trigger"),
        "samples_ok": report.get("samples_ok"),
        "samples_lost": report.get("samples_lost"),
        "loss_pct": report.get("loss_pct"),
        "button_to_visible_avg_ms": stats.get("avg_ms"),
        "button_to_visible_p50_ms": stats.get("p50_ms"),
        "button_to_visible_p95_ms": stats.get("p95_ms"),
        "button_to_visible_max_ms": stats.get("max_ms"),
        "button_to_visible_min_ms": stats.get("min_ms"),
        "button_to_visible_stddev_ms": stats.get("stddev_ms"),
    }
else:
    stats = report.get("rtt_stats") or {}
    summary = {
        "label": report.get("label"),
        "measurement": report.get("measurement"),
        "trigger": report.get("trigger"),
        "samples_ok": report.get("samples_ok"),
        "samples_lost": report.get("samples_lost"),
        "loss_pct": report.get("loss_pct"),
        "avg_rtt_ms": stats.get("avg_ms"),
        "p50_rtt_ms": stats.get("p50_ms"),
        "p95_rtt_ms": stats.get("p95_ms"),
        "max_rtt_ms": stats.get("max_ms"),
        "min_rtt_ms": stats.get("min_ms"),
        "stddev_rtt_ms": stats.get("stddev_ms"),
    }
summary

## Reading The Result

- In `rtt` mode, `p95_rtt_ms` is the best headline stability metric for the report.
- In `button_to_visible` mode, `button_to_visible_p95_ms` is the best headline button-to-returned-state latency metric.
- `avg_*_ms` gives the typical latency for the selected measurement mode.
- `max_*_ms` shows the worst spike seen in the run.
- `loss_pct` should ideally stay at `0.0`.